In [ ]:
import os
from dotenv import load_dotenv

import chromadb
import google.generativeai as genai

from sentence_transformers import SentenceTransformer

C:\Users\DELL\AppData\Local\Temp\ipykernel_22620\3174229761.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [14]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
#print(GOOGLE_API_KEY[:10])   # Just for testing

AQ.Ab8RN6K


In [15]:
llm = genai.GenerativeModel(
    "gemini-2.5-flash"
)

In [6]:
response = llm.generate_content("Say Hello")

print(response.text)

Hello!


In [16]:
response = llm.generate_content(
    "Say Hello"
)

print(response.text)

Hello!


In [17]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [18]:
client = chromadb.PersistentClient(
    path="../chroma_db"
)

collection = client.get_collection(
    "research_papers"
)

print(collection.count())

1000


In [19]:
def retrieve_documents(question: str, top_k: int = 5):
    """
    Retrieve the most relevant research papers from ChromaDB.

    Args:
        question (str): User query.
        top_k (int): Number of papers to retrieve.

    Returns:
        dict: ChromaDB query results.
    """

    # Convert question into embedding
    query_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True
    )

    # Search ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    return results

In [20]:
results = retrieve_documents("Explain Hawking Radiation")

In [21]:
results.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

In [22]:
for i, meta in enumerate(results["metadatas"][0], start=1):

    print("="*80)
    print(f"Paper {i}")
    print("="*80)

    print("Title :", meta["title"])
    print("Category :", meta["category"])
    print()

Paper 1
Title : Hawking radiation of linear dilaton black holes
Category : gr-qc

Paper 2
Title : A Single Trapped Ion as a Time-Dependent Harmonic Oscillator
Category : quant-ph

Paper 3
Title : Evolution of the Carter constant for inspirals into a black hole: effect
  of the black hole quadrupole
Category : gr-qc

Paper 4
Title : Particle propagation in cosmological backgrounds
Category : gr-qc

Paper 5
Title : Topology Change of Black Holes
Category : gr-qc



In [23]:
results = retrieve_documents(
    "Explain Hawking Radiation"
)


In [24]:
for i, meta in enumerate(results["metadatas"][0], start=1):
    print(meta["title"])

Hawking radiation of linear dilaton black holes
A Single Trapped Ion as a Time-Dependent Harmonic Oscillator
Evolution of the Carter constant for inspirals into a black hole: effect
  of the black hole quadrupole
Particle propagation in cosmological backgrounds
Topology Change of Black Holes


In [25]:
def build_context(results):
    """
    Convert retrieved papers into a formatted context string for the LLM.
    """

    context = ""

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    for i, (doc, meta) in enumerate(zip(docs, metas), start=1):

        context += f"""
================================================================================
Paper {i}
================================================================================

Title:
{meta["title"]}

Authors:
{meta["authors"]}

Category:
{meta["category"]}

Abstract:
{doc}

"""

    return context

In [26]:
context = build_context(results)

print(context[:2500])


Paper 1

Title:
Hawking radiation of linear dilaton black holes

Authors:
G. Clement, J.C. Fabris and G.T. Marques

Category:
gr-qc

Abstract:
Hawking radiation of linear dilaton black holes   We compute exactly the semi-classical radiation spectrum for a class of
non-asymptotically flat charged dilaton black holes, the so-called linear
dilaton black holes. In the high frequency regime, the temperature for these
black holes generically agrees with the surface gravity result. In the special
case where the black hole is massless, we show that, although the surface
gravity remains finite, there is no radiation, in agreement with the fact that
massless objects cannot radiate.



Paper 2

Title:
A Single Trapped Ion as a Time-Dependent Harmonic Oscillator

Authors:
Nicolas C. Menicucci and G. J. Milburn

Category:
quant-ph

Abstract:
A Single Trapped Ion as a Time-Dependent Harmonic Oscillator   We show how a single trapped ion may be used to test a variety of important
physical models rea

In [27]:
def build_prompt(question: str, context: str):
    """
    Create the prompt sent to Gemini.
    """

    prompt = f"""
You are an AI Research Assistant.

Your task is to answer the user's question ONLY using the retrieved research papers provided below.

Instructions:
- Use ONLY the information present in the context.
- Do NOT use outside knowledge.
- If the answer is not available in the retrieved papers, reply:
  "I couldn't find enough information in the retrieved papers."
- If multiple papers discuss the topic, combine their findings.
- Mention the paper titles whenever appropriate.
- Keep your answer clear, concise, and scientifically accurate.

================================================================================
Retrieved Research Papers
================================================================================

{context}

================================================================================
User Question
================================================================================

{question}

================================================================================
Answer
================================================================================
"""

    return prompt

In [28]:
question = "Explain Hawking Radiation."

prompt = build_prompt(question, context)

print(prompt[:3000])


You are an AI Research Assistant.

Your task is to answer the user's question ONLY using the retrieved research papers provided below.

Instructions:
- Use ONLY the information present in the context.
- Do NOT use outside knowledge.
- If the answer is not available in the retrieved papers, reply:
  "I couldn't find enough information in the retrieved papers."
- If multiple papers discuss the topic, combine their findings.
- Mention the paper titles whenever appropriate.
- Keep your answer clear, concise, and scientifically accurate.

Retrieved Research Papers


Paper 1

Title:
Hawking radiation of linear dilaton black holes

Authors:
G. Clement, J.C. Fabris and G.T. Marques

Category:
gr-qc

Abstract:
Hawking radiation of linear dilaton black holes   We compute exactly the semi-classical radiation spectrum for a class of
non-asymptotically flat charged dilaton black holes, the so-called linear
dilaton black holes. In the high frequency regime, the temperature for these
black holes gen

In [29]:
def generate_answer(prompt: str):
    """
    Send the prompt to Gemini and return the generated response.
    """

    response = llm.generate_content(prompt)

    return response.text

In [30]:
answer = generate_answer(prompt)

print(answer)

Hawking radiation is a semi-classical radiation spectrum (Hawking radiation of linear dilaton black holes). For a class of non-asymptotically flat charged dilaton black holes, known as linear dilaton black holes, the temperature of this radiation in the high frequency regime generally matches the surface gravity result (Hawking radiation of linear dilaton black holes). However, in the special case of a massless black hole, there is no radiation, despite a finite surface gravity, which aligns with the principle that massless objects cannot radiate (Hawking radiation of linear dilaton black holes).

Relatedly, Gibbons-Hawking radiation, described as thermal radiation in an expanding universe, has been proposed for simulation using a single trapped ion (A Single Trapped Ion as a Time-Dependent Harmonic Oscillator). While such simulations might yield a different spectrum from the actual Gibbons-Hawking case, they can still share important experimental signatures (A Single Trapped Ion as a 

In [31]:
def ask_question(question: str, top_k: int = 5):
    """
    Complete RAG pipeline.
    """

    results = retrieve_documents(question, top_k)

    context = build_context(results)

    prompt = build_prompt(question, context)

    answer = generate_answer(prompt)

    return answer

In [32]:
print(
    ask_question("Explain Hawking Radiation.")
)

Hawking radiation is a semi-classical radiation spectrum. For a class of non-asymptotically flat charged dilaton black holes, known as linear dilaton black holes, its temperature in the high frequency regime generally agrees with the surface gravity result. Notably, massless black holes, despite possessing a finite surface gravity, do not emit Hawking radiation, aligning with the principle that massless objects cannot radiate (Hawking radiation of linear dilaton black holes). It has also been described as thermal radiation, especially in the context of Gibbons-Hawking radiation in an expanding universe (A Single Trapped Ion as a Time-Dependent Harmonic Oscillator).
